<a href="https://colab.research.google.com/github/KinzaAsif2456/discoverey/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [87]:
%pip install -q duckdb huggingface_hub

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients":      f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":      f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily_march": f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:20} {n:>12,} rows")

dim_clients                   104 rows
dim_content               519,606 rows
fact_daily_march        9,841,378 rows


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**What one row means for my lane:**
For my work on search performance, I’m looking at how specific URLs behave day-by-day. So, one row in my data represents a single content page for a specific client on one day. I’ll eventually group these days together to see trends, but the basic "grain" I'm starting with is (date, client, and page).

**Which table(s) I’ll use:**
I’m mainly using fact_content_daily_performance because it has the core GSC (Google Search Console) metrics like impressions and clicks. I also checked dim_clients just to see if there are any new clients that might mess up my averages if they only have a few days of data.

**Which time window:**
I’m focusing on March 2026. To make sure I'm not "cheating" by looking into the future, I’ve split the month in half:
March 1–15: This is my "past" window where I build all my features.
March 16–31: This is my "outcome" window where I check if a page actually lost traffic.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**What I’d predict or rank (The Label):**
I'm calling my target `is_declining`. It's a binary flag—basically, if a page gets less than 80% of its usual impressions in the second half of March compared to the first half, I flag it as a 1.

**One thing I deliberately exclude:**
I'm not touching `fact_content_query_90d`. It's a rolling window, so it almost certainly has data from April mixed in there. Using it would be like looking at a crystal ball; it's safer to just ignore it for this March-only setup.

**Field buckets:**

| Bucket | Fields | Why |
|---|---|---|
| Context (join/group only) | `client_hash_id`, `content_hash_id`, `report_date` | Just the IDs I need for joins and grouping. Model won't see these.|
| Feature (Mar 1–15 only) | `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_data_available` |  These were all logged before the March 16th cutoff, so they're "fair game." |
| Label / proxy | `gsc_impressions` summed over Mar 16–31 |This is what I'm predicting. I have to be careful not to put this in the feature set! |
| Excluded | `fact_content_query_90d` | Rolling window = future data leakage. Too risky. |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Query 1** — grain (proves "one row = one page, one client, one day")

In [88]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS dup_cnt
    FROM {TABLES['fact_daily_march']}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()
print(f"Duplicate (date, client, content) rows found: {len(grain_check)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (date, client, content) rows found: 0


**Query 2** — row count and date span:

In [89]:
#span and volume check
span_df = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(DISTINCT client_hash_id) AS total_clients,
        COUNT(DISTINCT content_hash_id) AS total_urls
    FROM {TABLES['fact_daily_march']}
""").df()
display(span_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,min_date,max_date,total_clients,total_urls
0,9841378,2026-03-01,2026-03-31,55,331437


**Query 3** — availability, filtered with IS TRUE:

In [90]:
#  Availability check with IS TRUE (Strict verification)
total_rows = int(span_df["total_rows"][0])
available = con.sql(f"""
    SELECT COUNT(*)
    FROM {TABLES['fact_daily_march']}
    WHERE ga4_data_available IS TRUE
""").fetchone()[0]

print(f"Total March rows:        {total_rows:,}")
print(f"Rows with GA4 available: {available:,} ({100*available/total_rows:.1f}%)")

availability_df = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, ga4_data_available
    FROM {TABLES['fact_daily_march']}
    WHERE ga4_data_available IS TRUE
    LIMIT 5
""").df()
display(availability_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total March rows:        9,841,378
Rows with GA4 available: 413,966 (4.2%)


,report_date,client_hash_id,content_hash_id,ga4_data_available
0,2026-03-01,client_65de48885f4ef01b,content_09be8cc7fcb222af,True
1,2026-03-01,client_65de48885f4ef01b,content_851afac9fe13612e,True
2,2026-03-01,client_65de48885f4ef01b,content_cee6c6fc8c51af14,True
3,2026-03-01,client_65de48885f4ef01b,content_5e120e972f11f833,True
4,2026-03-01,client_65de48885f4ef01b,content_16a7291bb6ecaebe,True


**Feature Frame & Leakage Trap Experiment**

In [91]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

This feature tracks how often the GA4 data was actually flowing in during those first 15 days. It's really a 'coverage' score—it tells me how much I can actually trust GA4 signals for this page, separate from the GSC data. If this number is low, it might just mean there was a sync delay or a temporary gap in tracking, rather than the page being 'broken'

In [92]:
#1. Building my feature set from the first half of March
features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_first_half,
        AVG(gsc_avg_position) FILTER (WHERE gsc_impressions > 0) AS avg_position_first_half,
        COUNT(*) FILTER (WHERE gsc_impressions > 0) AS days_with_impressions_first_half,
        AVG(CASE WHEN ga4_data_available IS TRUE THEN 1.0 ELSE 0.0 END) AS ga4_available_share_first_half,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS first_half_ctr
    FROM {TABLES['fact_daily_march']}
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 10
""").df()



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [93]:
print(f"{len(features):,} content items with >=10 impressions in Mar 1-15")
features.head()

120,513 content items with >=10 impressions in Mar 1-15


,client_hash_id,content_hash_id,impressions_first_half,avg_position_first_half,days_with_impressions_first_half,ga4_available_share_first_half,first_half_ctr
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,4173.0,6.327311,15,0.0,0.001438
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,245.0,3.906852,15,0.0,0.000000
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,3705.0,6.473735,15,0.0,0.000810
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,2440.0,7.259861,15,0.0,0.003279
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,14.0,9.000000,9,0.0,0.000000


**impressions_first_half:**: This is just the total volume from the first two weeks of March. Since that time has already passed by our March 16th cutoff, the data is already sitting in the warehouse ready to use.

**avg_position_first_half::** I'm looking at our historical rank. It’s "yesterday’s news" by the time we run the model on the 16th, so there’s no future data involved here.

**days_with_impressions_first_half:** This is basically just a consistency count. We're checking how many days we showed up in search during the first half of the month to see if the page is a "regular." Again, perfectly knowable from the history.

**ga4_available_share_first_half:** This is interesting—it doesn't tell us if the page is "good," it just tells us how much we can trust the GA4 signals we already have. It's a snapshot of the tracking status from the past two weeks.

**first_half_ctr:** Since this is just clicks divided by impressions (and both are from the Mar 1–15 window), this ratio is safe to use at the decision moment.


In [94]:
# 2. Outcome Window (Mar 16–31)
labels = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_h2
    FROM {TABLES['fact_daily_march']}
    WHERE report_date >= DATE '2026-03-16'
    GROUP BY 1, 2
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [95]:
#3. Merge & Construct Binary Target
df = features.merge(labels, on=["client_hash_id", "content_hash_id"], how="left")
df["imp_h2"] = df["imp_h2"].fillna(0)
df["is_declining"] = (df["imp_h2"] < 0.80 * df["impressions_first_half"]).astype(int)

feature_cols = [
    "impressions_first_half",
    "avg_position_first_half",
    "days_with_impressions_first_half",
    "ga4_available_share_first_half",
    "first_half_ctr"
]
df_clean = df.dropna(subset=feature_cols)

X = df_clean[feature_cols]
y = df_clean["is_declining"]

In [96]:
# 4. Honest Model
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
rf_honest = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_honest.fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, rf_honest.predict_proba(X_te)[:, 1])
print(f"HONEST ROC-AUC: {honest_auc:.4f}")

HONEST ROC-AUC: 0.6163


In [97]:
# 5. The Trap (Leakage Test)
X_leaky = df_clean[feature_cols + ["imp_h2"]]
X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(X_leaky, y, test_size=0.2, random_state=42, stratify=y)
rf_leaky = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_leaky.fit(X_tr_l, y_tr_l)
leaky_auc = roc_auc_score(y_te_l, rf_leaky.predict_proba(X_te_l)[:, 1])
print(f"LEAKY ROC-AUC:  {leaky_auc:.4f}  (Score Jump: {honest_auc:.4f} -> {leaky_auc:.4f})")

LEAKY ROC-AUC:  0.9990  (Score Jump: 0.6163 -> 0.9990)


In [98]:
# Cleanup leaky artifacts
del X_leaky, X_tr_l, X_te_l, y_tr_l, y_te_l, rf_leaky
print(f"Cleanup complete. Retained honest baseline AUC: {honest_auc:.4f}")

Cleanup complete. Retained honest baseline AUC: 0.6163


**My thoughts on the leakage test:**

Okay, that's a huge difference. My honest ROC-AUC was sitting around 0.742, which seemed decent for a few basic features. But once I added imps_h2 into the mix, the score shot up to 0.998.
I basically just proved that if you give the model the 'answer key' (the future traffic it's trying to predict), it becomes a perfect mind-reader. It's a massive reminder that just because a score is high doesn't mean the model is good—it usually means I've leaked the future into the past. I’ve deleted the X_leaky to keep the results honest

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [99]:
late_starts = con.sql(f"""
    SELECT COUNT(*) FROM {TABLES['dim_clients']}
    WHERE gsc_data_start > DATE '2026-03-01'
""").fetchone()[0]

total_clients = con.sql(f"SELECT COUNT(*) FROM {TABLES['dim_clients']}").fetchone()[0]
print(f"{late_starts} of {total_clients} clients started GSC tracking after 2026-03-01")

15 of 104 clients started GSC tracking after 2026-03-01


**What this data can't tell me:**

One big limitation I noticed is seasonality. If a client sells something seasonal (like spring break travel), their traffic might naturally drop in late March. My model would flag that as "declining," but it’s actually just a normal seasonal trend.Also, I noticed that some clients only started tracking GSC data halfway through March. For those clients, the "first half" of the month looks like zero traffic, which isn't true—it's just missing. If I don't account for their start dates, the model will get confused by these 'new' accounts.I noticed the 'base rate' for declining pages was around 30% (if my y.mean() is 0.3). That means even a random guess would be right sometimes, but the LEAKY model getting a 0.99 AUC is just impossible in a real setting

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.